# EmoBERT v5 — Contexte multi-tours
**Mémoire M1 — Zinedine Hamadi & Sara Hadidi — Sorbonne Université 2025/2026**

Suite au retour de la prof (22/06) : concaténer 2-3 tours précédents au lieu d'un seul.

## Principe
Pour chaque exemple annoté (context=N-1, utterance=N), on cherche dans le corpus
complet une ligne avec le même conv_id où utterance == context (N-1), ce qui donne
le tour N-2. Résultat : [N-2] [SEP] [N-1] [SEP] [N]

⚠️ Prérequis : `corpus_edu_nettoye.csv` doit être sur Drive dans `memoire_M1/`
⚠️ GPU T4 requis. Utilise `Exécution` → `Tout exécuter`.

In [1]:
# ── Cellule 0 : Installation & imports ───────────────────────────────────────
!pip install -q transformers datasets torch scikit-learn matplotlib seaborn odfpy

from google.colab import drive
drive.mount('/content/drive')

import os, json, gc, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

TARGET_CLASSES = ['stress', 'frustration', 'engagement', 'confusion', 'satisfaction', 'neutre']
label2id = {c: i for i, c in enumerate(TARGET_CLASSES)}
id2label = {i: c for c, i in label2id.items()}
SEED     = 42
DRIVE    = '/content/drive/MyDrive/memoire_M1'
os.makedirs(DRIVE, exist_ok=True)

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('✓ Cellule 0 OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Mounted at /content/drive
GPU : Tesla T4
✓ Cellule 0 OK


In [2]:
# ── Cellule PATCH : désactiver torchvision dans datasets ─────────────────────
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False
ds_config.TORCHCODEC_AVAILABLE  = False

import sys
if 'torchvision' in sys.modules:
    del sys.modules['torchvision']

print('✓ torchvision désactivé dans datasets.config')

✓ torchvision désactivé dans datasets.config


## Étape 1 — Charger annotations + corpus complet

In [3]:
# ── Charger les annotations (avec conv_id) ────────────────────────────────────
df_z = pd.read_excel(f'{DRIVE}/zinedine_annotation.ods', engine='odf')
df_s = pd.read_excel(f'{DRIVE}/sara_annotation.ods',     engine='odf')

df_annot = pd.concat([df_z, df_s], ignore_index=True)
df_annot = df_annot[df_annot['edu_emotion'].notna()].copy()
df_annot['edu_emotion'] = df_annot['edu_emotion'].str.lower().str.strip()
df_annot['edu_emotion'] = df_annot['edu_emotion'].replace({
    'confision': 'confusion', 'neuret': 'neutre', 'neurtre': 'neutre'
})
df_annot = df_annot[df_annot['edu_emotion'].isin(TARGET_CLASSES)]
df_annot = df_annot.drop_duplicates(subset=['utterance']).reset_index(drop=True)

print(f'Annotations chargées : {len(df_annot)}')
print(f'Colonnes : {list(df_annot.columns)}')

# Vérifier qu'on a bien conv_id et context
assert 'conv_id' in df_annot.columns, 'conv_id manquant !'
assert 'context' in df_annot.columns, 'context manquant !'

# ── Charger le corpus complet (pour retrouver l'historique) ──────────────────
print('\nChargement corpus_edu_nettoye.csv...')
df_corpus = pd.read_csv(f'{DRIVE}/corpus_edu_nettoye.csv')
print(f'Corpus complet : {len(df_corpus)} lignes')
print(f'Colonnes : {list(df_corpus.columns)}')

Annotations chargées : 494
Colonnes : ['conv_id', 'context', 'utterance', 'emotion', 'matched_kw', 'source', 'edu_emotion', 'metacog_label', 'notes', 'annotator']

Chargement corpus_edu_nettoye.csv...
Corpus complet : 28516 lignes
Colonnes : ['source', 'conv_id', 'context', 'utterance', 'emotion', 'matched_kw']


## Étape 2 — Reconstruire le contexte multi-tours

In [4]:
# ── Construire un index conv_id → liste des (context, utterance) ─────────────
print('Indexation du corpus par conv_id...')
corpus_by_conv = {}
for _, row in df_corpus.iterrows():
    cid = row['conv_id']
    if cid not in corpus_by_conv:
        corpus_by_conv[cid] = []
    corpus_by_conv[cid].append({
        'context': str(row['context']).strip(),
        'utterance': str(row['utterance']).strip()
    })
print(f'✓ {len(corpus_by_conv)} conversations indexées')


def find_previous_turn(conv_id, current_context):
    """
    Cherche dans le corpus une ligne de la même conversation
    où utterance == current_context. Retourne son context (= N-2).
    """
    if conv_id not in corpus_by_conv:
        return None
    for pair in corpus_by_conv[conv_id]:
        if pair['utterance'] == current_context:
            return pair['context']
    return None


def build_multiturn_text(row, max_turns=3):
    """
    Construit le texte final avec jusqu'à max_turns tours de contexte.
    [N-2] [SEP] [N-1] [SEP] [N]
    """
    turns = []
    context_n1 = str(row['context']).strip()
    utterance_n = str(row['utterance']).strip()

    # Chercher N-2
    context_n2 = find_previous_turn(row['conv_id'], context_n1)

    if context_n2 and context_n2 not in ['nan', '']:
        turns.append(context_n2)
    if context_n1 and context_n1 not in ['nan', '']:
        turns.append(context_n1)
    turns.append(utterance_n)

    return ' [SEP] '.join(turns)


print('Construction du contexte multi-tours...')
df_annot['text_multiturn'] = df_annot.apply(build_multiturn_text, axis=1)

# Statistiques
n_with_2turns = (df_annot['text_multiturn'].str.count(r'\[SEP\]') == 2).sum()
n_with_1turn  = (df_annot['text_multiturn'].str.count(r'\[SEP\]') == 1).sum()
n_with_0turn  = (df_annot['text_multiturn'].str.count(r'\[SEP\]') == 0).sum()

print(f'\n✓ Construction terminée')
print(f'  Exemples avec 3 tours (N-2,N-1,N) : {n_with_2turns}')
print(f'  Exemples avec 2 tours (N-1,N)     : {n_with_1turn}')
print(f'  Exemples avec 1 tour (N seul)     : {n_with_0turn}')

print(f'\nExemple 3 tours :')
examples_3 = df_annot[df_annot['text_multiturn'].str.count(r'\[SEP\]') == 2]
if len(examples_3) > 0:
    print(f'  "{examples_3["text_multiturn"].iloc[0][:200]}"')

df_annot.to_csv(f'{DRIVE}/df_annot_multiturn.csv', index=False)
print(f'\n✓ df_annot_multiturn.csv sauvegardé sur Drive')

Indexation du corpus par conv_id...
✓ 21335 conversations indexées
Construction du contexte multi-tours...

✓ Construction terminée
  Exemples avec 3 tours (N-2,N-1,N) : 123
  Exemples avec 2 tours (N-1,N)     : 371
  Exemples avec 1 tour (N seul)     : 0

Exemple 3 tours :
  "Well done to you. Not everyone has that type of focus. What where you studying for? [SEP] Studying to be a pharmacist. [SEP] Thats great. Did you ace the test?"

✓ df_annot_multiturn.csv sauvegardé sur Drive


## Étape 3 — Enrichir avec GoEmotions (comme v4)

In [5]:
GO_EMOTIONS_MAP = {
    'nervousness': 'stress',    'fear':        'stress',
    'anger':       'frustration', 'annoyance': 'frustration',
    'disgust':     'frustration', 'disapproval':'frustration',
    'curiosity':   'engagement',  'excitement': 'engagement',
    'desire':      'engagement',  'admiration': 'engagement',
    'confusion':   'confusion',   'surprise':   'confusion',
    'realization': 'confusion',
    'joy':         'satisfaction','gratitude':  'satisfaction',
    'pride':       'satisfaction','relief':     'satisfaction',
    'approval':    'satisfaction',
    'neutral':     'neutre',
}

print('Chargement GoEmotions...')
go = load_dataset('google-research-datasets/go_emotions', 'simplified')
go_label_names = go['train'].features['labels'].feature.names

rows_go = []
for split in ['train', 'validation', 'test']:
    for row in go[split]:
        text   = row['text'].strip()
        labels = row['labels']
        if not text or not labels:
            continue
        go_label  = go_label_names[labels[0]]
        our_label = GO_EMOTIONS_MAP.get(go_label)
        if our_label and our_label != 'neutre':
            rows_go.append({'text_multiturn': text, 'edu_emotion': our_label})

df_go = pd.DataFrame(rows_go)

TARGET_PER_CLASS = 400
rows_sampled = []
for classe in TARGET_CLASSES:
    if classe == 'neutre':
        continue
    subset = df_go[df_go['edu_emotion'] == classe]
    n = min(TARGET_PER_CLASS, len(subset))
    if n > 0:
        rows_sampled.append(subset.sample(n, random_state=SEED))

df_go_balanced = pd.concat(rows_sampled, ignore_index=True)
df_go_balanced.to_csv(f'{DRIVE}/df_go_balanced_v5.csv', index=False)
print(f'GoEmotions : {len(df_go_balanced)} exemples')
print(df_go_balanced['edu_emotion'].value_counts().to_string())
print('\n✓ Étape 3 OK')

Chargement GoEmotions...


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

GoEmotions : 2000 exemples
edu_emotion
stress          400
frustration     400
engagement      400
confusion       400
satisfaction    400

✓ Étape 3 OK


## Étape 4 — Sous-échantillonner neutre + fusionner

In [6]:
# Sous-échantillonner neutre dans les annotations
df_neutre     = df_annot[df_annot['edu_emotion'] == 'neutre'].sample(60, random_state=SEED)
df_non_neutre = df_annot[df_annot['edu_emotion'] != 'neutre']
df_balanced   = pd.concat([df_non_neutre, df_neutre], ignore_index=True)

print('Distribution annotations rééquilibrées :')
print(df_balanced['edu_emotion'].value_counts().to_string())

# Fusion finale : annotations multi-tours + GoEmotions (single-turn)
df_final = pd.concat([
    df_balanced[['text_multiturn', 'edu_emotion']],
    df_go_balanced[['text_multiturn', 'edu_emotion']]
], ignore_index=True)

df_final = df_final.rename(columns={'text_multiturn': 'text'})
df_final = df_final.drop_duplicates(subset='text').reset_index(drop=True)

print('\n=== Distribution finale v5 ===')
print(df_final['edu_emotion'].value_counts().to_string())
print(f'Total : {len(df_final)}')

df_final.to_csv(f'{DRIVE}/df_final_v5.csv', index=False)
print('\n✓ df_final_v5.csv sauvegardé — Étape 4 OK')

Distribution annotations rééquilibrées :
edu_emotion
engagement      127
neutre           60
satisfaction     34
frustration      24
confusion        23
stress           10

=== Distribution finale v5 ===
edu_emotion
engagement      527
satisfaction    434
frustration     424
confusion       422
stress          409
neutre           60
Total : 2276

✓ df_final_v5.csv sauvegardé — Étape 4 OK


## Étape 5 — Réentraîner EmoBERT v5

In [7]:
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding, EarlyStoppingCallback
)

MODEL_BASE = 'j-hartmann/emotion-english-distilroberta-base'
tokenizer  = RobertaTokenizer.from_pretrained(MODEL_BASE)

if 'df_final' not in globals():
    df_final = pd.read_csv(f'{DRIVE}/df_final_v5.csv')
    print('df_final rechargé depuis Drive')

X = df_final['text'].tolist()
y = [label2id[e] for e in df_final['edu_emotion'].tolist()]

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_v,  X_te,  y_v,  y_te  = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp)
print(f'Train: {len(X_tr)} | Val: {len(X_v)} | Test: {len(X_te)}')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=320)  # plus long pour multi-tours

hf_tr = Dataset.from_dict({'text': X_tr, 'label': y_tr})
hf_v  = Dataset.from_dict({'text': X_v,  'label': y_v})
hf_te = Dataset.from_dict({'text': X_te, 'label': y_te})

hf_tr = hf_tr.map(tokenize, batched=True, remove_columns=['text'])
hf_v  = hf_v.map(tokenize,  batched=True, remove_columns=['text'])
hf_te = hf_te.map(tokenize, batched=True, remove_columns=['text'])

for ds in [hf_tr, hf_v, hf_te]:
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_BASE, num_labels=len(TARGET_CLASSES),
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(list(range(len(TARGET_CLASSES)))),
    y=y_tr
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to('cuda')
print(f'Class weights : {dict(zip(TARGET_CLASSES, class_weights.round(2)))}')

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred[0], axis=-1)
    return {
        'accuracy': round(accuracy_score(eval_pred[1], preds), 4),
        'f1_macro': round(f1_score(eval_pred[1], preds, average='macro', zero_division=0), 4)
    }

training_args = TrainingArguments(
    output_dir='emobert_v5_output',
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    seed=SEED,
    report_to='none',
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get('labels')
        outputs = model(**inputs)
        logits  = outputs.get('logits')
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=hf_tr, eval_dataset=hf_v,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Réentraînement EmoBERT v5 (contexte multi-tours + class weights)...')
trainer.train()
print('✓ Étape 5 OK')

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Étape 5 OK


## Étape 6 — Évaluation + sauvegarde

In [8]:
predictions = trainer.predict(hf_te)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

acc = accuracy_score(labels, preds)
f1  = f1_score(labels, preds, average='macro', zero_division=0)

print(f'\n=== Résultats EmoBERT v5 (contexte multi-tours) ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-macro : {f1:.4f}')
print(f'\n{classification_report(labels, preds, target_names=TARGET_CLASSES, zero_division=0)}')

print('=== Historique des versions ===')
print(f'v1 → F1: 0.20 (corpus déséquilibré)')
print(f'v2 → F1: 0.58 (GoEmotions)')
print(f'v3 → F1: 0.53-0.57 (+ augmentation / class weights)')
print(f'v4 → F1: 0.59 (+ context simple + modèle spécialisé)')
print(f'v5 → F1: {f1:.2f} (+ contexte multi-tours N-2,N-1,N)')

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, ax=ax)
ax.set_title(f'Matrice de confusion — EmoBERT v5 (F1={f1:.2f})')
plt.tight_layout()
plt.savefig(f'{DRIVE}/confusion_matrix_v5.png', dpi=150, bbox_inches='tight')
plt.show()

import shutil
trainer.save_model('emobert_v5')
tokenizer.save_pretrained('emobert_v5')
with open('emobert_v5/label_mapping.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

drive_dest = '/content/drive/MyDrive/emobert_v5'
if os.path.exists(drive_dest):
    shutil.rmtree(drive_dest)
shutil.copytree('emobert_v5', drive_dest)

print(f'\n✓ EmoBERT v5 sauvegardé sur Drive : {drive_dest}')
print('✓ Étape 6 OK — Terminé !')


=== Résultats EmoBERT v5 (contexte multi-tours) ===
Accuracy : 0.6272
F1-macro : 0.5910

              precision    recall  f1-score   support

      stress       0.78      0.76      0.77        41
 frustration       0.68      0.79      0.73        43
  engagement       0.83      0.36      0.50        53
   confusion       0.60      0.64      0.62        42
satisfaction       0.66      0.63      0.64        43
      neutre       0.17      0.83      0.29         6

    accuracy                           0.63       228
   macro avg       0.62      0.67      0.59       228
weighted avg       0.70      0.63      0.63       228

=== Historique des versions ===
v1 → F1: 0.20 (corpus déséquilibré)
v2 → F1: 0.58 (GoEmotions)
v3 → F1: 0.53-0.57 (+ augmentation / class weights)
v4 → F1: 0.59 (+ context simple + modèle spécialisé)
v5 → F1: 0.59 (+ contexte multi-tours N-2,N-1,N)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ EmoBERT v5 sauvegardé sur Drive : /content/drive/MyDrive/emobert_v5
✓ Étape 6 OK — Terminé !


In [9]:
# Tester différents seuils pour neutre
neutre_idx = TARGET_CLASSES.index('neutre')
probs_all = torch.softmax(
    torch.tensor(predictions.predictions), dim=-1
).numpy()

for threshold in [0.5, 0.6, 0.7, 0.8]:
    preds_thresh = []
    for prob in probs_all:
        pred_idx = prob.argmax()
        if pred_idx == neutre_idx and prob[pred_idx] < threshold:
            # Prendre la 2e émotion la plus probable
            prob_copy = prob.copy()
            prob_copy[neutre_idx] = 0
            pred_idx = prob_copy.argmax()
        preds_thresh.append(pred_idx)

    f1_t = f1_score(labels, preds_thresh, average='macro', zero_division=0)
    print(f'Seuil neutre = {threshold} → F1-macro = {f1_t:.4f}')

Seuil neutre = 0.5 → F1-macro = 0.5905
Seuil neutre = 0.6 → F1-macro = 0.6039
Seuil neutre = 0.7 → F1-macro = 0.5941
Seuil neutre = 0.8 → F1-macro = 0.5966
